In [1]:
#cell 1: (2e) đọc manifest.csv, chuẩn hóa landmark (trừ cổ tay, chia cỡ bàn tay) + vận tốc, gom thành tensor train/val/test trên CUDA
import numpy as np, pandas as pd, torch
from pathlib import Path

assert torch.cuda.is_available(), "Chưa bật CUDA"
dev = "cuda"
BASE = Path(r"C:\Users\Acer Nitro 5 Tiger\Downloads\Data_321")
m = pd.read_csv(BASE / "manifest.csv")

def feat(f):
    lm = np.load(BASE / f)["lm"]                                  # (T, 21, 3), mất tay = NaN
    lm = lm - lm[:, :1]                                           # trừ cổ tay -> không phụ thuộc vị trí trong ảnh
    lm = lm / np.linalg.norm(lm[:, 9], axis=1)[:, None, None]     # chia cỡ bàn tay (cổ tay -> gốc ngón giữa)
    pos = lm.reshape(len(lm), 63)
    vel = np.vstack([np.zeros((1, 63)), np.diff(pos, axis=0)])    # vận tốc = hiệu 2 khung liền nhau
    has = ~np.isnan(pos[:, 0])                                    # khung nào có tay
    return np.nan_to_num(np.hstack([pos, vel])).astype(np.float32), has

Tmax = int(m.khung.max())
def build(df):
    X = np.zeros((len(df), Tmax, 126), np.float32); M = np.zeros((len(df), Tmax), bool)
    for i, f in enumerate(df.path):
        x, h = feat(f); X[i, :len(x)] = x; M[i, :len(x)] = h      # khung đệm thêm và khung mất tay: M = False
    return torch.tensor(X, device=dev), torch.tensor(df.nhan.values, device=dev), torch.tensor(M, device=dev)

meta = {t: m[m.tap == t].reset_index(drop=True) for t in ["train", "val", "test"]}
data = {t: build(meta[t]) for t in meta}

print("Tmax:", Tmax)
for t, (X, y, M) in data.items():
    print(t, "| X:", tuple(X.shape), "| nhãn paper/rock/scissors:", torch.bincount(y, minlength=3).tolist(),
          "| khung có tay: %.1f%%" % (100 * M.sum().item() / meta[t].khung.sum()),
          "| có NaN:", torch.isnan(X).any().item())

Tmax: 355
train | X: (151, 355, 126) | nhãn paper/rock/scissors: [53, 49, 49] | khung có tay: 99.7% | có NaN: False
val | X: (31, 355, 126) | nhãn paper/rock/scissors: [11, 10, 10] | khung có tay: 97.7% | có NaN: False
test | X: (33, 355, 126) | nhãn paper/rock/scissors: [11, 11, 11] | khung có tay: 99.0% | có NaN: False


In [2]:
#cell 2: (2e) định nghĩa 3 mô hình dự đoán theo từng khung (chỉ nhìn các khung trước): TCN, GRU nhỏ, Transformer nhỏ
import torch, torch.nn as nn, torch.nn.functional as F

class Block(nn.Module):                                    # 1 khối TCN: conv giãn nở, chỉ đệm bên trái để không nhìn tương lai
    def __init__(s, c, d):
        super().__init__(); s.pad = 2 * d; s.conv = nn.Conv1d(c, c, 3, dilation=d); s.drop = nn.Dropout(0.1)
    def forward(s, x):
        return x + s.drop(F.relu(s.conv(F.pad(x, (s.pad, 0)))))

class TCN(nn.Module):
    def __init__(s, c=64):
        super().__init__(); s.inp = nn.Linear(126, c)
        s.blocks = nn.Sequential(*[Block(c, d) for d in (1, 2, 4, 8, 16)])    # nhìn lại tối đa 63 khung
        s.out = nn.Linear(c, 3)
    def forward(s, x):                                      # x: (B, T, 126)
        return s.out(s.blocks(s.inp(x).transpose(1, 2)).transpose(1, 2))      # (B, T, 3)

class GRU(nn.Module):
    def __init__(s, c=64):
        super().__init__(); s.inp = nn.Linear(126, c); s.rnn = nn.GRU(c, c, batch_first=True); s.out = nn.Linear(c, 3)
    def forward(s, x):
        return s.out(s.rnn(F.relu(s.inp(x)))[0])

class Trans(nn.Module):
    def __init__(s, c=64, T=Tmax):
        super().__init__(); s.inp = nn.Linear(126, c); s.pos = nn.Parameter(torch.randn(1, T, c) * 0.02)
        s.enc = nn.TransformerEncoder(nn.TransformerEncoderLayer(c, 4, 128, 0.1, batch_first=True), 2)
        s.out = nn.Linear(c, 3)
    def forward(s, x):
        T = x.shape[1]; mask = nn.Transformer.generate_square_subsequent_mask(T, device=x.device)   # khung t chỉ nhìn khung <= t
        return s.out(s.enc(s.inp(x) + s.pos[:, :T], mask=mask))

# kiểm tra: chạy thử trên CUDA + đổi dữ liệu từ khung 100 trở đi, kết quả các khung trước đó phải giữ nguyên
x = data["val"][0][:4]
for name, cls in [("TCN", TCN), ("GRU", GRU), ("Transformer", Trans)]:
    net = cls().to(dev).eval()
    with torch.no_grad():
        a = net(x); x2 = x.clone(); x2[:, 100:] = torch.randn_like(x2[:, 100:]); b = net(x2)
    print(name, "| tham số:", sum(p.numel() for p in net.parameters()), "| output:", tuple(a.shape),
          "| không nhìn tương lai:", torch.allclose(a[:, :100], b[:, :100], atol=1e-5))

TCN | tham số: 70083 | output: (4, 355, 3) | không nhìn tương lai: True
GRU | tham số: 33283 | output: (4, 355, 3) | không nhìn tương lai: True
Transformer | tham số: 97987 | output: (4, 355, 3) | không nhìn tương lai: True


In [3]:
#cell 3: (2e) hàm train dùng chung: CUDA, tối đa 60 epoch, early stopping theo val loss, lưu checkpoint tốt nhất, lần sau tự nạp lại
import time, torch, torch.nn.functional as F

CK = BASE / "checkpoints"; CK.mkdir(exist_ok=True)

def loss_acc(net, X, y, M):                                  # loss và accuracy trung bình trên các khung có tay (M = True)
    out = net(X)                                             # (B, T, 3)
    l = F.cross_entropy(out.transpose(1, 2), y[:, None].expand(-1, X.shape[1]), reduction="none")   # (B, T)
    ok = out.argmax(-1) == y[:, None]
    return (l * M).sum() / M.sum(), (ok & M).sum() / M.sum()

def train(name, cls, epochs=60, bs=16, patience=10, lai=False):
    torch.manual_seed(0)
    net = cls().to(dev); f = CK / f"{name}.pt"
    if f.exists() and not lai:                               # đã có checkpoint -> nạp lại, không train
        ck = torch.load(f, map_location=dev); net.load_state_dict(ck["w"])
        print(f"{name}: nạp checkpoint (epoch {ck['epoch']}, val_loss {ck['val_loss']:.4f}) từ {f}. Muốn train lại: lai=True")
        return net.eval(), ck["hist"]
    opt = torch.optim.AdamW(net.parameters(), 1e-3, weight_decay=1e-2)
    Xt, yt, Mt = data["train"]; best, wait, hist, t0 = 1e9, 0, [], time.time()
    for ep in range(1, epochs + 1):
        net.train(); perm = torch.randperm(len(Xt), device=dev); tl = 0
        for i in range(0, len(Xt), bs):
            b = perm[i:i + bs]; l, _ = loss_acc(net, Xt[b], yt[b], Mt[b])
            opt.zero_grad(); l.backward(); opt.step(); tl += l.item() * len(b)
        net.eval()
        with torch.no_grad(): vl, va = loss_acc(net, *data["val"])
        hist.append(dict(ep=ep, train_loss=tl / len(Xt), val_loss=vl.item(), val_acc=va.item()))
        if vl.item() < best:                                 # val loss tốt nhất từ trước tới giờ -> lưu checkpoint
            best, wait = vl.item(), 0
            torch.save(dict(w=net.state_dict(), epoch=ep, val_loss=best, hist=hist), f)
        else:
            wait += 1
        if ep % 5 == 0 or wait >= patience or ep == epochs:
            h = hist[-1]; print(f"{name} ep{ep:02d} | train_loss {h['train_loss']:.4f} | val_loss {h['val_loss']:.4f} | val_acc {h['val_acc']:.3f}")
        if wait >= patience: print(f"{name}: dừng sớm ở epoch {ep} (val loss không giảm {patience} epoch liên tiếp)"); break
    ck = torch.load(f, map_location=dev); net.load_state_dict(ck["w"])
    print(f"{name}: xong sau {time.time() - t0:.0f}s | tốt nhất ở epoch {ck['epoch']} (val_loss {ck['val_loss']:.4f}) | checkpoint: {f}")
    return net.eval(), hist

print("Checkpoint sẽ lưu tại:", CK)

Checkpoint sẽ lưu tại: C:\Users\Acer Nitro 5 Tiger\Downloads\Data_321\checkpoints


In [4]:
#cell 4: (2e) train lần lượt TCN, GRU nhỏ, Transformer nhỏ và tóm tắt kết quả trên tập val
nets, hists = {}, {}
for name, cls in [("TCN", TCN), ("GRU", GRU), ("Transformer", Trans)]:
    nets[name], hists[name] = train(name, cls)
    print("-" * 80)

for name, h in hists.items():
    b = min(h, key=lambda r: r["val_loss"])
    print(f"{name:12s} | epoch tốt nhất {b['ep']:2d}/{len(h)} | val_loss {b['val_loss']:.4f} | val_acc (mọi khung) {b['val_acc']:.3f}")

TCN ep05 | train_loss 0.3568 | val_loss 0.3751 | val_acc 0.891
TCN ep10 | train_loss 0.2559 | val_loss 0.3146 | val_acc 0.895
TCN ep15 | train_loss 0.2132 | val_loss 0.4371 | val_acc 0.792
TCN ep20 | train_loss 0.2040 | val_loss 0.7245 | val_acc 0.799
TCN: dừng sớm ở epoch 20 (val loss không giảm 10 epoch liên tiếp)
TCN: xong sau 2s | tốt nhất ở epoch 10 (val_loss 0.3146) | checkpoint: C:\Users\Acer Nitro 5 Tiger\Downloads\Data_321\checkpoints\TCN.pt
--------------------------------------------------------------------------------
GRU ep05 | train_loss 0.6544 | val_loss 0.5755 | val_acc 0.763
GRU ep10 | train_loss 0.2557 | val_loss 0.4167 | val_acc 0.847
GRU ep15 | train_loss 0.2297 | val_loss 0.5646 | val_acc 0.842
GRU ep18 | train_loss 0.2044 | val_loss 0.6456 | val_acc 0.826
GRU: dừng sớm ở epoch 18 (val loss không giảm 10 epoch liên tiếp)
GRU: xong sau 1s | tốt nhất ở epoch 8 (val_loss 0.3469) | checkpoint: C:\Users\Acer Nitro 5 Tiger\Downloads\Data_321\checkpoints\GRU.pt
----------

In [5]:
#cell 5: (2e) đánh giá 3 mô hình trên tập test: mỗi ngưỡng tin cậy 0.5->0.9, tỉ lệ video có khung vừa tự tin >= ngưỡng vừa đoán đúng + thời điểm đoán đúng đầu tiên (ms)
import pandas as pd, torch

THR = [0.5, 0.6, 0.7, 0.8, 0.9]
X, y, M = data["test"]
rows = []
for name, net in nets.items():
    with torch.no_grad(): conf, pred = net(X).softmax(-1).max(-1)        # (B, T): độ tin cậy và nhãn đoán của từng khung
    for t in THR:
        ok = (pred == y[:, None]) & (conf >= t) & M                       # khung đúng, đủ tự tin, có tay
        has = ok.any(1)
        ms = ok.float().argmax(1)[has].float() * 1000 / 120               # khung đúng đầu tiên -> ms (120 fps)
        rows.append(dict(model=name, nguong=t, ti_le=has.float().mean().item(),
                         ms_trung_vi=ms.median().item() if has.any() else float("nan")))

r = pd.DataFrame(rows)
pa = r.pivot(index="model", columns="nguong", values="ti_le") * 100
pa["TB"] = pa.mean(1)
print("Tỉ lệ video đoán đúng và đủ tự tin (%), tập test:"); print(pa.round(1))
print("\nThời điểm đoán đúng đầu tiên, trung vị (ms, tính từ đầu clip):")
print(r.pivot(index="model", columns="nguong", values="ms_trung_vi").round(0))

Tỉ lệ video đoán đúng và đủ tự tin (%), tập test:
nguong         0.5    0.6    0.7    0.8   0.9    TB
model                                              
GRU          100.0  100.0  100.0  100.0  84.8  97.0
TCN          100.0  100.0   97.0   93.9  93.9  97.0
Transformer   97.0   97.0   97.0   97.0  97.0  97.0

Thời điểm đoán đúng đầu tiên, trung vị (ms, tính từ đầu clip):
nguong        0.5   0.6   0.7    0.8    0.9
model                                      
GRU          25.0  42.0  67.0  108.0  100.0
TCN           8.0  25.0  33.0   67.0  133.0
Transformer   0.0   0.0   0.0    0.0    8.0


C:\Users\Acer Nitro 5 Tiger\AppData\Local\Temp\ipykernel_18404\2894582937.py:18: Pandas4Warning: Starting with pandas version 4.0 all arguments of mean will be keyword-only.
  pa["TB"] = pa.mean(1)


In [6]:
#cell 6: (2e) đường cong theo thời gian trên tập test: tại mỗi mốc khung, % video đoán đúng và độ tin cậy trung bình của 3 mô hình
import pandas as pd, torch

X, y, M = data["test"]
T = [0, 1, 2, 3, 4, 6, 8, 12, 16, 24, 32, 48, 64, 96]          # chỉ số khung (120 fps)
ms = [round(t * 1000 / 120) for t in T]                         # đổi ra ms tính từ đầu clip
acc, cf = {}, {}
for name, net in nets.items():
    with torch.no_grad(): conf, pred = net(X).softmax(-1).max(-1)
    a, c = [], []
    for t in T:
        v = M[:, t]                                              # video còn khung t và có tay
        n = max(v.sum().item(), 1)
        a.append(((pred[:, t] == y) & v).sum().item() / n * 100)
        c.append(conf[:, t][v].mean().item() * 100)
    acc[name], cf[name] = a, c

print("% video đoán đúng tại từng mốc (đoán bừa = 33.3%):"); print(pd.DataFrame(acc, index=ms).rename_axis("ms").round(1))
print("\nĐộ tin cậy trung bình (%) tại từng mốc:");              print(pd.DataFrame(cf, index=ms).rename_axis("ms").round(1))

% video đoán đúng tại từng mốc (đoán bừa = 33.3%):
       TCN    GRU  Transformer
ms                            
0     68.8   65.6         65.6
8     68.8   68.8         68.8
17    78.1   71.9         65.6
25    75.0   71.9         71.9
33    72.7   75.8         69.7
50    72.7   75.8         72.7
67    75.8   75.8         72.7
100   75.8   78.8         72.7
133   78.8   78.8         72.7
200   81.8   81.8         78.8
267   90.9   84.8         81.8
400   90.9   90.9         81.8
533   93.3   93.3         86.7
800  100.0  100.0         95.8

Độ tin cậy trung bình (%) tại từng mốc:
      TCN   GRU  Transformer
ms                          
0    53.5  42.0         81.1
8    58.6  50.2         81.2
17   63.9  58.0         86.9
25   65.0  64.1         87.4
33   70.1  68.0         87.2
50   71.8  75.0         87.3
67   76.4  79.6         87.2
100  79.4  84.6         86.8
133  81.8  86.6         87.5
200  82.9  87.6         87.1
267  87.3  89.2         90.5
400  93.0  91.6         91.3
533  9

In [7]:
#cell 7: (2e) khi mô hình nói "tin cậy >= ngưỡng" thì đúng bao nhiêu % (độ chính xác) và bao nhiêu % khung đạt ngưỡng (độ phủ); tập test, tách cả clip và nửa đầu clip
import pandas as pd, torch

THR = [0.5, 0.6, 0.7, 0.8, 0.9]
X, y, M = data["test"]
L = torch.tensor(meta["test"].khung.values, device=dev)[:, None]      # độ dài từng video
nua = torch.arange(X.shape[1], device=dev)[None, :] < L / 2            # True ở nửa đầu của mỗi video
vung = {"cả clip": M, "nửa đầu": M & nua}

out = {}
for name, net in nets.items():
    with torch.no_grad(): out[name] = net(X).softmax(-1).max(-1)       # (độ tin cậy, nhãn đoán) từng khung

P, C = {}, {}
for v, base in vung.items():
    for name, (conf, pred) in out.items():
        ok = pred == y[:, None]
        sel = [(conf >= t) & base for t in THR]                         # khung có tay, đủ tự tin, thuộc vùng đang xét
        P[(v, name)] = [((ok & s).sum() / s.sum().clamp(min=1)).item() * 100 for s in sel]
        C[(v, name)] = [(s.sum() / base.sum()).item() * 100 for s in sel]

idx = pd.MultiIndex.from_tuples(list(P.keys()), names=["vùng", "model"])
print("Độ chính xác (%) trong các khung có tin cậy >= ngưỡng:"); print(pd.DataFrame(list(P.values()), index=idx, columns=THR).round(1))
print("\nĐộ phủ (%): tỉ lệ khung đạt ngưỡng:");                  print(pd.DataFrame(list(C.values()), index=idx, columns=THR).round(1))

Độ chính xác (%) trong các khung có tin cậy >= ngưỡng:
                      0.5   0.6   0.7   0.8   0.9
vùng    model                                    
cả clip TCN          93.3  94.9  95.6  96.1  98.1
        GRU          91.7  92.4  93.0  93.6  94.5
        Transformer  85.7  88.9  92.6  97.2  98.9
nửa đầu TCN          88.1  90.6  91.8  92.4  95.8
        GRU          84.9  86.1  86.9  87.7  88.9
        Transformer  78.1  83.5  89.4  95.4  98.5

Độ phủ (%): tỉ lệ khung đạt ngưỡng:
                      0.5   0.6   0.7   0.8   0.9
vùng    model                                    
cả clip TCN          94.7  90.3  87.0  81.6  73.7
        GRU          96.9  94.5  91.8  88.1  68.7
        Transformer  98.4  94.3  89.4  83.9  79.1
nửa đầu TCN          90.1  82.7  77.2  71.5  58.7
        GRU          94.1  89.5  85.0  78.9  57.3
        Transformer  97.0  89.4  81.9  75.4  67.6


## Tổng kết Task 2: so sánh 3 mô hình (đánh giá trên tập test, 33 video)

**Kết quả chính:** cả 3 mô hình đều vượt mốc 80% ở thước đo trong Decision.docx (trung bình 5 ngưỡng đều 97.0%). Tuy vậy thước đo đó khá dễ, nên ta nhìn thêm bảng ở cell 7: *khi mô hình nói "tin cậy ≥ ngưỡng" thì nó đúng bao nhiêu %*.

| Độ chính xác (%) khi tin cậy ≥ ngưỡng | 0.5 | 0.6 | 0.7 | 0.8 | 0.9 |
|---|---|---|---|---|---|
| TCN, cả clip | 93.3 | 94.9 | 95.6 | 96.1 | 98.1 |
| GRU, cả clip | 91.7 | 92.4 | 93.0 | 93.6 | 94.5 |
| Transformer, cả clip | 85.7 | 88.9 | 92.6 | 97.2 | 98.9 |
| TCN, nửa đầu clip | 88.1 | 90.6 | 91.8 | 92.4 | 95.8 |
| GRU, nửa đầu clip | 84.9 | 86.1 | 86.9 | 87.7 | 88.9 |
| Transformer, nửa đầu clip | 78.1 | 83.5 | 89.4 | 95.4 | 98.5 |

### Nhận xét từng mô hình

- **TCN: ổn định nhất.** Đúng nhiều nhất ở ngưỡng thấp (0.5 đến 0.6, hai ngưỡng ta ưu tiên) và tăng đều theo ngưỡng. Ở nửa đầu clip vẫn đạt 88% ở ngưỡng 0.5. Là lựa chọn an toàn nếu muốn ra quyết định sớm.
- **Transformer: "chắc mới nói".** Ở ngưỡng thấp thì hay tự tin mà sai (chỉ 78% ở nửa đầu clip, ngưỡng 0.5). Nhưng từ ngưỡng 0.8 trở lên lại chính xác nhất (95 đến 99%) và còn nhiều khung đạt ngưỡng 0.9 nhất (67.6% ở nửa đầu clip). Nếu dùng ngưỡng cao thì rất mạnh, nếu dùng ngưỡng thấp thì kém.
- **GRU: yếu nhất.** Độ chính xác tăng rất ít khi ngưỡng tăng (nửa đầu clip chỉ từ 84.9 lên 88.9), nên độ tin cậy của nó ít giúp phân biệt lần nào chắc, lần nào không.

### Cân nhắc khi chọn mô hình đầu tư thêm

- Ta ưu tiên ngưỡng 0.5 và 0.6, nên **TCN** là ứng viên số một.
- **Transformer** đáng giữ lại để thử, nếu game chấp nhận chờ tới ngưỡng cao (≥ 0.8) mới ra chiêu.
- Chênh lệch giữa các mô hình chỉ khoảng vài phần trăm, trên 33 video, nên chưa đủ để loại hẳn mô hình nào chỉ bằng bảng này.

### Điều cần lưu ý (đem sang Task 3)

1. Ngay ở 0 ms (khung đầu clip), cả 3 mô hình đã đúng khoảng 66% (đoán bừa là 33%). Chưa rõ vì tay đã có dáng gần giống cử chỉ cuối hay vì mô hình học lối tắt. Cần kiểm tra.
2. Tập test chỉ có 33 video, và clip trong train/val/test đến từ cùng hai người quay. Kết quả chưa nói được mô hình có chạy tốt với người chơi mới hay camera khác hay không.
3. Số ms trong các bảng tính từ đầu clip, chưa phải từ lúc người chơi bắt đầu ra tay trong game.